# Jacobian lens — OLMo + Gemma walkthrough

Steps 1–3 of `walkthrough.ipynb`, run on two locally fitted lenses (`output/`)
over local model checkpoints:

- **OLMo**: `olmo2-0425-1B-sft` copy at `/workspace/models/olmo2_1B/olmo2_1b_base_sft`
- **Gemma**: `gemma-3-1b-it` copy at `/workspace/models/gemma3_1B/gemma3_1b_ancestor`

In [1]:
import jlens

jlens.configure_logging()

CONFIGS = {
    "olmo": {
        "model_path": "/workspace/models/olmo2_1B/olmo2_1b_base_sft",
        "lens_path": "../output/olmo2_1b_base_sft_jacobian_lens.pt",
    },
    "gemma": {
        "model_path": "/workspace/models/gemma3_1B/gemma3_1b_ancestor",
        "lens_path": "../output/gemma3_1b_ancestor_jacobian_lens.pt",
    },
}

## 1. Load the models

`jlens.from_hf` wraps an already-loaded HuggingFace model into the `LensModel`
interface. Both models are ~1B so they fit on the GPU side by side.

In [2]:
import torch
import transformers

models = {}
tokenizers = {}
for name, cfg in CONFIGS.items():
    hf_model = transformers.AutoModelForCausalLM.from_pretrained(
        cfg["model_path"], dtype=torch.bfloat16
    ).cuda()
    tokenizers[name] = transformers.AutoTokenizer.from_pretrained(cfg["model_path"])
    models[name] = jlens.from_hf(hf_model, tokenizers[name])
models

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

{'olmo': HFLensModel(Olmo2ForCausalLM, n_layers=16, d_model=2048),
 'gemma': HFLensModel(Gemma3ForCausalLM, n_layers=26, d_model=1152)}

## 2. Load the pre-fitted lenses

`JacobianLens.from_pretrained` also accepts a local `.pt` path. The lens holds
one `[d_model, d_model]` matrix per layer.

In [3]:
lenses = {
    name: jlens.JacobianLens.from_pretrained(cfg["lens_path"])
    for name, cfg in CONFIGS.items()
}
lenses

{'olmo': JacobianLens(d_model=2048, n_prompts=419, source_layers=[0..14] (15 layers)),
 'gemma': JacobianLens(d_model=1152, n_prompts=460, source_layers=[0..24] (25 layers))}

## 3. Apply: J-lens vs logit lens

`lens.apply(model, prompt, positions=...)` runs one forward pass, transports
each layer's residual into the final-layer basis with `J_l`, and decodes
through the model's own unembedding. `use_jacobian=False` skips the transport —
that's the vanilla logit lens.

Both models are instruction-tuned, so the question is wrapped in each model's
chat template with a generation prompt appended, and read out at the **last**
position — where the model predicts the first token of its answer.

One tokenization gotcha: the templated string already starts with the BOS
token, but `model.encode` tokenizes with `add_special_tokens=True`, which for
Gemma would prepend a second `<bos>`. Flipping `add_bos_token` off fixes that
(OLMo's tokenizer never auto-adds anything); the assert checks that `encode`
reproduces the chat template's own tokenization exactly.

In [4]:
question = "What is the capital of the state containing Minneapolis? Respond with only the city's name."
messages = [{"role": "user", "content": question}]


def top5(tokenizer, logits):
    return [tokenizer.decode([t]) for t in logits.topk(5).indices]


for name, model in models.items():
    tokenizer = tokenizers[name]
    lens = lenses[name]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    tokenizer.add_bos_token = False  # prompt already starts with BOS
    assert (
        model.encode(prompt)[0].tolist()
        == tokenizer(prompt, add_special_tokens=False).input_ids
    )

    layers = [
        model.n_layers // 4,
        model.n_layers // 2,
        model.n_layers // 4 * 3,
        model.n_layers - 2,
    ]

    jlens_logits, model_logits, _ = lens.apply(
        model, prompt, layers=layers, positions=[-1]
    )
    logit_lens, _, _ = lens.apply(
        model, prompt, layers=layers, positions=[-1], use_jacobian=False
    )

    print(f"=== {name} ({model.n_layers} layers) ===")
    print(f"prompt: {prompt!r}")
    for layer in layers:
        print(f"L{layer:>3} logit-lens: {top5(tokenizer, logit_lens[layer][0])}")
        print(f"L{layer:>3} J-lens:     {top5(tokenizer, jlens_logits[layer][0])}")
    print(f"model:           {top5(tokenizer, model_logits[0])}")
    print()

=== olmo (16 layers) ===
prompt: "<|endoftext|><|user|>\nWhat is the capital of the state containing Minneapolis? Respond with only the city's name.\n<|assistant|>\n"
L  4 logit-lens: [' building', ' thorough', 'iana', ' if', 'aws']
L  4 J-lens:     ['s', 't', 'l', 'e', 'y']
L  8 logit-lens: [' among', ' depending', 'wick', ' Cong', ' collaboration']
L  8 J-lens:     ['<|endoftext|>', '.\n', '.\n\n', '.\n\n\n', '.\n\n\n\n\n']
L 12 logit-lens: [' capital', ' capitals', 'abeth', ' cities', '-capital']
L 12 J-lens:     [' Seattle', ' Minneapolis', ' Chicago', ' Boston', ' Milwaukee']
L 14 logit-lens: ['Min', 'St', 'C', 'H', 'The']
L 14 J-lens:     [' Minneapolis', 'Min', 'Minnesota', 'St', 'Saint']
model:           ['Min', 'The', 'Saint', 'St', 'C']

=== gemma (26 layers) ===
prompt: "<bos><start_of_turn>user\nWhat is the capital of the state containing Minneapolis? Respond with only the city's name.<end_of_turn>\n<start_of_turn>model\n"
L  6 logit-lens: ['1', '3', '4', '6', '9']
L  6 J-l